In [1]:
import warnings
warnings.filterwarnings('ignore')

import os
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
from tqdm import tqdm
import pickle
import boto3
import datetime as dt
from tqdm import tqdm
import shutil

try:
    import optbinning
except:
    ! pip install optbinning

try:
    import catboost
except:
    ! pip install catboost

In [2]:
dtm_now = dt.datetime.now()
print(f'Latest run date: {dtm_now}')

Latest run date: 2025-03-10 20:49:53.477424


#### Constants

In [3]:
str_project = os.getcwd().split('/')[4].replace('_','-')
print(f'Project: {str_project}')

str_task = os.getcwd().split('/')[5]
print(f'Task: {str_task}')

# output
str_dirname_output = './output'

Project: 20250307-funded-trends
Task: 07_get_predictions


#### Make output dir

In [4]:
try:
    os.mkdir(str_dirname_output)
except:
    pass

#### Import raw data

In [5]:
str_filename = 'df.gzip'
str_uri = f's3://{str_project}/04_join_targets/{str_filename}'
df = pd.read_parquet(
    str_uri,
)
# show
df

,accountid,request_datetime,response_model_name,file_key,bitdebtor,bitdebtor__app,dealerstate__app,strdealershiptrackertype__app,strname__app,bitdealertrack__app,...,fltNetChgOff_2,fltNetChgOff_3,fltNetChgOff_6,fltNetChgOff_12,fltNetChgOff_24,co_at_60,co_at_90,co_at_180,co_at_360,co_at_720
0,5714239,2021-07-26,Gen10,deprecated/03_pull_payloads_tbldove/df_request...,1,1,Utah,Franchise,Utah,True,...,0.0,0.0,0.0,0.0,7827.16,0.0,0.0,0.0,0.0,0.315976
1,5713063,2021-07-26,Gen10,deprecated/03_pull_payloads_tbldove/df_request...,1,1,Illinois,Franchise,Illinois,True,...,0.0,0.0,0.0,0.0,0.00,0.0,0.0,0.0,0.0,0.000000
2,5702434,2021-07-26,Gen10,deprecated/03_pull_payloads_tbldove/df_request...,1,1,Iowa,Franchise,Iowa,True,...,0.0,0.0,0.0,0.0,0.00,0.0,0.0,0.0,0.0,0.000000
3,5704330,2021-07-27,Gen10,deprecated/03_pull_payloads_tbldove/df_request...,0,0,Idaho,Franchise,Idaho,True,...,0.0,0.0,0.0,0.0,0.00,0.0,0.0,0.0,0.0,0.000000
4,5704330,2021-07-27,Gen10,deprecated/03_pull_payloads_tbldove/df_request...,1,1,Idaho,Franchise,Idaho,True,...,0.0,0.0,0.0,0.0,0.00,0.0,0.0,0.0,0.0,0.000000
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
100026,8705652,2025-03-07,PRESTIGE-GEN-XIII,nan,1,1,Michigan,Franchise,Michigan,False,...,0.0,0.0,0.0,0.0,0.00,0.0,0.0,0.0,0.0,0.000000
100027,8676945,2025-03-07,PRESTIGE-GEN-XIII,nan,1,1,Pennsylvania,Independent,New Jersey,True,...,0.0,0.0,0.0,0.0,0.00,0.0,0.0,0.0,0.0,0.000000
100028,8712238,2025-03-07,PRESTIGE-GEN-XIII,nan,0,0,Oklahoma,Franchise,Kansas,True,...,0.0,0.0,0.0,0.0,0.00,0.0,0.0,0.0,0.0,0.000000
100029,8605449,2025-03-07,PRESTIGE-GEN-XIII,nan,1,1,Ohio,Franchise,Ohio,True,...,0.0,0.0,0.0,0.0,0.00,0.0,0.0,0.0,0.0,0.000000


#### Get Gen 12 predictions

In [6]:
list_cols = [
    'gen12_pd',
    'gen12_lgd',
]
str_filename = 'df.gzip'
str_uri = f's3://{str_project}/05_get_gen12_predictions/{str_filename}'
df_tmp = pd.read_parquet(
    str_uri,
    columns=list_cols,
)
# show
df_tmp

,gen12_pd,gen12_lgd
0,0.226593,0.635248
1,0.208254,0.742858
2,0.037012,0.676405
3,0.284456,0.656422
4,0.172819,0.631861
...,...,...
100026,0.129768,0.672289
100027,0.102756,0.622361
100028,0.068506,0.595623
100029,0.152090,0.602908


#### Concat horizontally

In [7]:
df = pd.concat([df, df_tmp], axis=1)
df

,accountid,request_datetime,response_model_name,file_key,bitdebtor,bitdebtor__app,dealerstate__app,strdealershiptrackertype__app,strname__app,bitdealertrack__app,...,fltNetChgOff_6,fltNetChgOff_12,fltNetChgOff_24,co_at_60,co_at_90,co_at_180,co_at_360,co_at_720,gen12_pd,gen12_lgd
0,5714239,2021-07-26,Gen10,deprecated/03_pull_payloads_tbldove/df_request...,1,1,Utah,Franchise,Utah,True,...,0.0,0.0,7827.16,0.0,0.0,0.0,0.0,0.315976,0.226593,0.635248
1,5713063,2021-07-26,Gen10,deprecated/03_pull_payloads_tbldove/df_request...,1,1,Illinois,Franchise,Illinois,True,...,0.0,0.0,0.00,0.0,0.0,0.0,0.0,0.000000,0.208254,0.742858
2,5702434,2021-07-26,Gen10,deprecated/03_pull_payloads_tbldove/df_request...,1,1,Iowa,Franchise,Iowa,True,...,0.0,0.0,0.00,0.0,0.0,0.0,0.0,0.000000,0.037012,0.676405
3,5704330,2021-07-27,Gen10,deprecated/03_pull_payloads_tbldove/df_request...,0,0,Idaho,Franchise,Idaho,True,...,0.0,0.0,0.00,0.0,0.0,0.0,0.0,0.000000,0.284456,0.656422
4,5704330,2021-07-27,Gen10,deprecated/03_pull_payloads_tbldove/df_request...,1,1,Idaho,Franchise,Idaho,True,...,0.0,0.0,0.00,0.0,0.0,0.0,0.0,0.000000,0.172819,0.631861
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
100026,8705652,2025-03-07,PRESTIGE-GEN-XIII,nan,1,1,Michigan,Franchise,Michigan,False,...,0.0,0.0,0.00,0.0,0.0,0.0,0.0,0.000000,0.129768,0.672289
100027,8676945,2025-03-07,PRESTIGE-GEN-XIII,nan,1,1,Pennsylvania,Independent,New Jersey,True,...,0.0,0.0,0.00,0.0,0.0,0.0,0.0,0.000000,0.102756,0.622361
100028,8712238,2025-03-07,PRESTIGE-GEN-XIII,nan,0,0,Oklahoma,Franchise,Kansas,True,...,0.0,0.0,0.00,0.0,0.0,0.0,0.0,0.000000,0.068506,0.595623
100029,8605449,2025-03-07,PRESTIGE-GEN-XIII,nan,1,1,Ohio,Franchise,Ohio,True,...,0.0,0.0,0.00,0.0,0.0,0.0,0.0,0.000000,0.152090,0.602908


#### Get Gen 13 Predictions

In [8]:
str_filename = 'df.gzip'
str_uri = f's3://{str_project}/06_get_gen13_predictions/{str_filename}'
df_tmp = pd.read_parquet(
    str_uri,
)
# get columns we want
list_cols = [col for col in df_tmp.columns if '_contribution' in col]
list_cols = list_cols + ['gen13_pd', 'gen13_lgd']
# subset
df_tmp = df_tmp[list_cols].copy()

# show
df_tmp

,fltgrossmonthly__income_sum_binned_contribution,au20s__tu_binned_contribution,miles_odometer__app_binned_contribution,rtl_trd__tu_binned_contribution,ENG-loan_to_value_binned_contribution,g002s__tu_binned_contribution,rev322__tu_binned_contribution,balmag01__tu_binned_contribution,g232s__tu_binned_contribution,inquirybanking12month__ln_binned_contribution,...,linkf098__tu_binned_contribution,linka006__tu_binned_contribution,inquiryshortterm12month__ln_binned_contribution,addrinputsubjectcount__ln_binned_contribution,g251c__tu_binned_contribution,g095s__tu_binned_contribution,s209a__tu_binned_contribution,inquirynonshortterm12month__ln_binned_contribution,gen13_pd,gen13_lgd
0,-0.079370,0.070605,0.068391,0.069732,0.047108,-0.044553,0.035855,-0.028432,-0.152658,-0.075483,...,0.004268,0.034744,-0.040978,-0.014258,-0.128272,-0.142548,-0.037731,-0.061136,0.353818,0.390043
1,0.016821,0.070605,0.022159,0.069732,0.047108,0.080829,0.035855,0.080222,0.067989,-0.075483,...,0.004268,-0.065002,-0.040978,-0.014258,-0.128272,0.059176,0.018359,0.033394,0.439522,0.416230
2,0.065652,-0.081654,0.022159,-0.098345,0.047108,0.080829,0.035855,-0.116947,0.200271,-0.075483,...,0.004268,0.034744,-0.040978,0.014632,0.022693,-0.142548,0.018359,0.033394,0.578671,0.323245
3,0.065652,-0.007667,-0.020006,0.069732,0.047108,-0.044553,0.035855,0.046765,-0.077534,-0.075483,...,0.004268,-0.081562,-0.040978,0.014632,0.022693,0.059176,-0.037731,0.033394,0.475007,0.390043
4,0.016821,-0.153846,-0.020006,0.069732,0.047108,-0.044553,0.035855,-0.116947,0.081609,-0.075483,...,0.004268,0.034744,0.147799,0.014632,0.022693,0.059176,0.018359,0.033394,0.494334,0.390043
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
100026,-0.035258,-0.137101,0.022159,0.069732,-0.099068,-0.044553,0.035855,0.046765,0.067989,-0.075483,...,0.004268,-0.018353,-0.040978,-0.014258,0.022693,0.059176,0.018359,0.033394,0.418451,0.337281
100027,-0.182637,-0.201805,0.012613,0.003788,0.047108,-0.044553,0.035855,-0.296269,-0.022883,-0.075483,...,0.004268,0.006152,0.147799,-0.013710,0.002340,0.059176,-0.037731,-0.061136,0.248432,0.390043
100028,-0.182637,-0.007667,0.022159,-0.052971,0.047108,-0.044553,-0.166417,-0.116947,0.081609,-0.075483,...,0.004268,0.034744,-0.040978,0.014632,0.022693,0.059176,0.018359,0.033394,0.310440,0.390043
100029,-0.079370,-0.081654,0.022159,-0.098345,0.001648,-0.044553,-0.166417,0.013170,0.134302,0.090501,...,0.004268,0.034744,-0.040978,-0.014258,0.022693,-0.142548,-0.120138,0.033394,0.348000,0.289997


#### Concat horizontally

In [9]:
df = pd.concat([df, df_tmp], axis=1)
df

,accountid,request_datetime,response_model_name,file_key,bitdebtor,bitdebtor__app,dealerstate__app,strdealershiptrackertype__app,strname__app,bitdealertrack__app,...,linkf098__tu_binned_contribution,linka006__tu_binned_contribution,inquiryshortterm12month__ln_binned_contribution,addrinputsubjectcount__ln_binned_contribution,g251c__tu_binned_contribution,g095s__tu_binned_contribution,s209a__tu_binned_contribution,inquirynonshortterm12month__ln_binned_contribution,gen13_pd,gen13_lgd
0,5714239,2021-07-26,Gen10,deprecated/03_pull_payloads_tbldove/df_request...,1,1,Utah,Franchise,Utah,True,...,0.004268,0.034744,-0.040978,-0.014258,-0.128272,-0.142548,-0.037731,-0.061136,0.353818,0.390043
1,5713063,2021-07-26,Gen10,deprecated/03_pull_payloads_tbldove/df_request...,1,1,Illinois,Franchise,Illinois,True,...,0.004268,-0.065002,-0.040978,-0.014258,-0.128272,0.059176,0.018359,0.033394,0.439522,0.416230
2,5702434,2021-07-26,Gen10,deprecated/03_pull_payloads_tbldove/df_request...,1,1,Iowa,Franchise,Iowa,True,...,0.004268,0.034744,-0.040978,0.014632,0.022693,-0.142548,0.018359,0.033394,0.578671,0.323245
3,5704330,2021-07-27,Gen10,deprecated/03_pull_payloads_tbldove/df_request...,0,0,Idaho,Franchise,Idaho,True,...,0.004268,-0.081562,-0.040978,0.014632,0.022693,0.059176,-0.037731,0.033394,0.475007,0.390043
4,5704330,2021-07-27,Gen10,deprecated/03_pull_payloads_tbldove/df_request...,1,1,Idaho,Franchise,Idaho,True,...,0.004268,0.034744,0.147799,0.014632,0.022693,0.059176,0.018359,0.033394,0.494334,0.390043
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
100026,8705652,2025-03-07,PRESTIGE-GEN-XIII,nan,1,1,Michigan,Franchise,Michigan,False,...,0.004268,-0.018353,-0.040978,-0.014258,0.022693,0.059176,0.018359,0.033394,0.418451,0.337281
100027,8676945,2025-03-07,PRESTIGE-GEN-XIII,nan,1,1,Pennsylvania,Independent,New Jersey,True,...,0.004268,0.006152,0.147799,-0.013710,0.002340,0.059176,-0.037731,-0.061136,0.248432,0.390043
100028,8712238,2025-03-07,PRESTIGE-GEN-XIII,nan,0,0,Oklahoma,Franchise,Kansas,True,...,0.004268,0.034744,-0.040978,0.014632,0.022693,0.059176,0.018359,0.033394,0.310440,0.390043
100029,8605449,2025-03-07,PRESTIGE-GEN-XIII,nan,1,1,Ohio,Franchise,Ohio,True,...,0.004268,0.034744,-0.040978,-0.014258,0.022693,-0.142548,-0.120138,0.033394,0.348000,0.289997


#### Save to s3

In [10]:
str_filename = 'df.gzip'
str_uri = f's3://{str_project}/{str_task}/{str_filename}'
df.to_parquet(
    str_uri,
    compression='gzip',
)